# Optimización MILP: Resolviendo el TSP en Cabárceno

En este notebook abordamos el núcleo matemático del problema. Modelaremos la visita al parque como un **Problema del Viajante (TSP)** sobre un grafo asimétrico.

Nuestro objetivo es encontrar la matriz de decisión $x_{ij} \in \{0,1\}$ que minimice la distancia total recorrida:

$$\min \sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij} x_{ij}$$

Donde $c_{ij}$ es la distancia real en metros (calculada vía Dijkstra sobre el grafo de OpenStreetMap) entre el recinto $i$ y el recinto $j$. Utilizaremos **Google OR-Tools** para resolver el sistema mediante Programación Lineal Entera Mixta (MILP).

In [1]:
import osmnx as ox
import networkx as nx
import numpy as np
import folium
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import geopandas as gpd
from shapely.ops import nearest_points
from pyproj import Transformer
from itertools import product as iproduct
import webbrowser, os
from collections import defaultdict

In [2]:
# Cargamos el grafo proyectado en metros (UTM) guardado
G_metros = ox.load_graphml('../data/processed/cabarceno_graph.graphml')

print(f"Grafo cargado. Nodos: {len(G_metros.nodes)}, Aristas: {len(G_metros.edges)}")
print(f"CRS: {G_metros.graph['crs']}")

Grafo cargado. Nodos: 1929, Aristas: 2678
CRS: EPSG:32630


## 1. Mapeo de Recintos a Nodos del Grafo

Definimos las coordenadas GPS (Longitud, Latitud) de los animales que queremos visitar hoy y buscamos el nodo matemático más cercano en nuestra red de carreteras.

In [3]:
# Buscamos los recintos del parque usando los tags que aparecen en OSM:
# praderas (meadow), matorrales (scrub), edificios (building) y atracciones turísticas
tags_recintos = {
    'landuse': ['meadow', 'grass'],
    'natural': 'scrub',
    'building': True,
    'tourism': 'attraction'
}

areas_gdf = ox.features_from_place("Parque de la Naturaleza de Cabárceno, Cantabria, Spain", tags_recintos)

# Nos quedamos solo con los que tienen nombre
recintos_nombrados = areas_gdf[areas_gdf['name'].notna()].copy()

# Si hay nombres duplicados, añadimos sufijo numérico para distinguirlos (ej. "Llama", "Llama (2)")
counts = {}
nombres_unicos = []
for nombre in recintos_nombrados['name']:
    counts[nombre] = counts.get(nombre, 0) + 1
    nombres_unicos.append(nombre)

seen = {}
nombres_finales = []
for nombre in nombres_unicos:
    if counts[nombre] > 1:
        seen[nombre] = seen.get(nombre, 0) + 1
        nombres_finales.append(f"{nombre} ({seen[nombre]})")
    else:
        nombres_finales.append(nombre)

recintos_nombrados = recintos_nombrados.copy()
recintos_nombrados['name'] = nombres_finales

print(f"Recintos con nombre encontrados: {len(recintos_nombrados)}")
print(recintos_nombrados['name'].tolist())


Recintos con nombre encontrados: 40
['Aula de Educación Medioambiental', 'Gorilas', 'Vaca y Caballo Monchino', 'Llama (1)', 'Jaguar', 'Gaur', 'Fauna Ibérica', 'Camello Bactriano', 'Gorila', 'Llama (2)', 'Rinoceronte Blanco', 'Tigre', 'Vaca Tudanca', 'Asno Somalí', 'Cebra', 'Hipopótamo', 'Jirafa, Avestruz y Eland', 'Addax y Búfalo Cafre', 'Bisonte Europeo', 'Cebras Grevy', 'Elefante Africano, Búfalo de agua y Cobo de Lechwe', 'Oryx del Cabo', 'Watusi (1)', 'Hipopótamos pigmeos', 'Oso', 'Cobos de agua', 'León', 'Lobo', 'Papion de Guinea', 'Wallaby de Bennet y Emú', 'Watusi (2)', 'Leones Marinos', 'Aves rapaces', 'Granja', 'Guepardo', 'Linces', 'Reptilario', 'Centro de Recuperación de Fauna Silvestre', 'Exhibiciones Aves rapaces', 'Casa de cebras Grevy']


In [4]:
# Descargamos todos los estacionamientos dentro del parque
parkings_gdf = ox.features_from_place(
    "Parque de la Naturaleza de Cabárceno, Cantabria, Spain",
    tags={'amenity': 'parking'}
)

# Calculamos el centroide de cada parking en UTM
parkings_proj = parkings_gdf.to_crs(G_metros.graph['crs'])
parkings_centroides_utm = parkings_proj.copy()
parkings_centroides_utm.geometry = parkings_proj.geometry.centroid

# print(f"Estacionamientos encontrados: {len(parkings_gdf)}")
# parkings_centroides_utm.to_crs("EPSG:4326")[['geometry']]


In [ ]:
# --- Entrada Principal: parking OSM way 358294677 ---
entrada_centroide_utm = parkings_centroides_utm.loc[('way', 358294677), 'geometry']

# --- Para cada recinto: parking más cercano SIEMPRE incluido,
#     más todos los parkings cuya distancia real borde-a-borde al recinto sea ≤ x m ---
recintos_proj = recintos_nombrados.to_crs(G_metros.graph['crs'])

# candidatos_utm[nombre] = lista de (x_utm, y_utm) de los puntos del parking más cercanos al recinto
candidatos_utm = {"Entrada Principal": [(entrada_centroide_utm.x, entrada_centroide_utm.y)]}

for nombre_recinto, geom_recinto in zip(recintos_nombrados['name'], recintos_proj.geometry):
    pts = []
    nearest_dist = float('inf')
    nearest_pt = None

    for geom_parking in parkings_proj.geometry:
        dist_borde = geom_recinto.distance(geom_parking)
        _, punto = nearest_points(geom_recinto, geom_parking)
        pt = (punto.x, punto.y)
        if dist_borde <= 30:
            pts.append(pt)
        if dist_borde < nearest_dist:
            nearest_dist = dist_borde
            nearest_pt = pt

    # El parking más cercano siempre se incluye, aunque esté a más de 50 m
    if nearest_pt not in pts:
        pts.insert(0, nearest_pt)

    candidatos_utm[nombre_recinto] = pts

nombres = list(candidatos_utm.keys())

Nodos únicos de carretera candidatos: 54
  📍 Entrada Principal: 1 parking(s) candidato(s)
  🦁 Aula de Educación Medioambiental: 1 parking(s) candidato(s)
  🦁 Gorilas: 1 parking(s) candidato(s)
  🦁 Vaca y Caballo Monchino: 3 parking(s) candidato(s)
  🦁 Llama (1): 1 parking(s) candidato(s)
  🦁 Jaguar: 1 parking(s) candidato(s)
  🦁 Gaur: 1 parking(s) candidato(s)
  🦁 Fauna Ibérica: 1 parking(s) candidato(s)
  🦁 Camello Bactriano: 1 parking(s) candidato(s)
  🦁 Gorila: 2 parking(s) candidato(s)
  🦁 Llama (2): 1 parking(s) candidato(s)
  🦁 Rinoceronte Blanco: 1 parking(s) candidato(s)
  🦁 Tigre: 2 parking(s) candidato(s)
  🦁 Vaca Tudanca: 2 parking(s) candidato(s)
  🦁 Asno Somalí: 1 parking(s) candidato(s)
  🦁 Cebra: 1 parking(s) candidato(s)
  🦁 Hipopótamo: 1 parking(s) candidato(s)
  🦁 Jirafa, Avestruz y Eland: 4 parking(s) candidato(s)
  🦁 Addax y Búfalo Cafre: 1 parking(s) candidato(s)
  🦁 Bisonte Europeo: 1 parking(s) candidato(s)
  🦁 Cebras Grevy: 1 parking(s) candidato(s)
  🦁 Elefante

In [ ]:
# Todos los puntos UTM únicos (orden de inserción, sin duplicados) → nodos del grafo
seen_set = set()
todos_pts = []
for pts in candidatos_utm.values():
    for pt in pts:
        if pt not in seen_set:
            seen_set.add(pt)
            todos_pts.append(pt)

x_all = [pt[0] for pt in todos_pts]
y_all = [pt[1] for pt in todos_pts]
nodos_all = ox.distance.nearest_nodes(G_metros, X=x_all, Y=y_all)
pt_to_nodo = {pt: nodo for pt, nodo in zip(todos_pts, nodos_all)}

In [ ]:
# candidatos_nodos[nombre] = lista de nodos del grafo, deduplicados, preservando orden
candidatos_nodos = {}
for nombre, pts in candidatos_utm.items():
    seen_n = set()
    nodos_uniq = []
    for pt in pts:
        nd = pt_to_nodo[pt]
        if nd not in seen_n:
            seen_n.add(nd)
            nodos_uniq.append(nd)
    candidatos_nodos[nombre] = nodos_uniq

In [ ]:
# Pre-computamos Dijkstra entre todos los nodos candidatos únicos
nodos_unicos = list(set(nd for ns in candidatos_nodos.values() for nd in ns))
dist_entre_nodos = {}
paths_entre_nodos = {}
for src in nodos_unicos:
    lengths, paths = nx.single_source_dijkstra(G_metros, src, weight='length')
    for dst in nodos_unicos:
        if dst in lengths:
            dist_entre_nodos[(src, dst)] = int(lengths[dst])
            paths_entre_nodos[(src, dst)] = paths[dst]

In [ ]:
# Transformer UTM → WGS84 para los marcadores del mapa
transformer = Transformer.from_crs(G_metros.graph['crs'], "EPSG:4326", always_xy=True)
pt_to_lonlat = {pt: transformer.transform(pt[0], pt[1]) for pt in todos_pts}

total_combos = 1
for nombre in nombres:
    total_combos *= len(candidatos_nodos[nombre])

print(f"Nodos únicos de carretera candidatos: {len(nodos_unicos)}")
for nombre in nombres:
    n_c = len(candidatos_nodos[nombre])
    print(f"  {'📍' if nombre == 'Entrada Principal' else '🦁'} {nombre}: {n_c} parking(s) candidato(s)")
print(f"\nDistancias pre-calculadas. Combinaciones totales a evaluar: {total_combos}")

## 2. Reducción del Problema: Matriz de Distancias Exactas

El *solver* no necesita conocer todas las curvas de Cabárceno, solo necesita un **grafo completo** con los costes reales $c_{ij}$ entre nuestros 5 puntos de interés. Calculamos estos costes en metros aplicando el algoritmo de **Dijkstra**.

In [6]:
# Mostramos la matriz de distancias base: usando el parking más cercano a cada recinto (candidato 0)
combo_base = [candidatos_nodos[n][0] for n in nombres]
n_base = len(combo_base)
matrix_base = np.zeros((n_base, n_base), dtype=int)
for i, ni in enumerate(combo_base):
    for j, nj in enumerate(combo_base):
        if i != j:
            matrix_base[i][j] = dist_entre_nodos.get((ni, nj), -1)

print("Matriz de distancias base (en metros), parking más cercano a cada recinto:")
print(matrix_base)


Matriz de distancias base (en metros), parking más cercano a cada recinto:
[[   0 2309 1462 ...  255 2283 3906]
 [2664    0 2206 ... 2789  222 2014]
 [1871 1954    0 ... 1996 1929 3552]
 ...
 [ 230 2512 1665 ...    0 2486 4110]
 [2618  256 2160 ... 2743    0 1968]
 [3891 1606 3433 ... 4016 1580    0]]


## 3. Resolución del TSP con Google OR-Tools

Instanciamos el modelo de enrutamiento. Configuramos la función de coste y aplicamos la metaheurística de Búsqueda Local Guiada (Guided Local Search) para escapar de mínimos locales y encontrar el orden óptimo de visita.

In [7]:
# Listas de nodos candidatos por enclosure, en el orden de 'nombres'
listas_candidatos = [candidatos_nodos[n] for n in nombres]

total_combos = 1
for lst in listas_candidatos:
    total_combos *= len(lst)

print(f"Evaluando {total_combos} combinación(es) de asignación de parkings...\n")

# --- FASE 1: barrido rápido con PATH_CHEAPEST_ARC para encontrar la mejor combinación ---
best_distance_fase1 = float('inf')
best_nodos_combo = None

def _solve_tsp(nodos_combo, use_gls=False):
    n = len(nodos_combo)
    dm = np.zeros((n, n), dtype=int)
    for i, ni in enumerate(nodos_combo):
        for j, nj in enumerate(nodos_combo):
            if i != j:
                dm[i][j] = dist_entre_nodos.get((ni, nj), 10**8)

    mgr = pywrapcp.RoutingIndexManager(n, 1, 0)
    mdl = pywrapcp.RoutingModel(mgr)

    def _cb(from_idx, to_idx, _dm=dm, _mgr=mgr):
        return _dm[_mgr.IndexToNode(from_idx)][_mgr.IndexToNode(to_idx)]

    cb_idx = mdl.RegisterTransitCallback(_cb)
    mdl.SetArcCostEvaluatorOfAllVehicles(cb_idx)

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    if use_gls:
        params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
        params.time_limit.seconds = 10

    sol = mdl.SolveWithParameters(params)
    if not sol:
        return None, None

    dist = sol.ObjectiveValue()
    order = []
    idx = mdl.Start(0)
    while not mdl.IsEnd(idx):
        order.append(mgr.IndexToNode(idx))
        idx = sol.Value(mdl.NextVar(idx))
    order.append(mgr.IndexToNode(idx))
    return dist, order

for combo_num, combo in enumerate(iproduct(*listas_candidatos), start=1):
    nodos_combo = list(combo)
    dist, _ = _solve_tsp(nodos_combo, use_gls=False)
    is_best = dist is not None and dist < best_distance_fase1
    if is_best:
        best_distance_fase1 = dist
        best_nodos_combo = nodos_combo
    print(f"  [{combo_num}/{total_combos}] dist={dist if dist is not None else -1} m {'✅ nuevo mejor' if is_best else ''}")

# --- FASE 2: GLS sobre la mejor combinación encontrada ---
print(f"\n🔍 Optimizando la mejor combinación con GLS...")
best_distance, best_order = _solve_tsp(best_nodos_combo, use_gls=True)

# Exportamos para las celdas de renderizado
optimal_order = best_order
nodos_objetivo = best_nodos_combo

print(f"\n✅ Mejor distancia total encontrada: {best_distance} metros")
print(f"Orden de visita (índices): {optimal_order}")
print(" -> ".join(nombres[i] for i in optimal_order))


Evaluando 27648 combinación(es) de asignación de parkings...

  [1/27648] dist=23084 m ✅ nuevo mejor
  [2/27648] dist=23069 m ✅ nuevo mejor
  [3/27648] dist=23085 m 
  [4/27648] dist=23070 m 
  [5/27648] dist=23084 m 
  [6/27648] dist=23069 m 
  [7/27648] dist=23085 m 
  [8/27648] dist=23070 m 
  [9/27648] dist=23084 m 
  [10/27648] dist=23069 m 
  [11/27648] dist=23085 m 
  [12/27648] dist=23070 m 
  [13/27648] dist=23084 m 
  [14/27648] dist=23069 m 
  [15/27648] dist=23085 m 
  [16/27648] dist=23070 m 
  [17/27648] dist=23084 m 
  [18/27648] dist=23069 m 
  [19/27648] dist=23084 m 
  [20/27648] dist=23069 m 
  [21/27648] dist=23084 m 
  [22/27648] dist=23069 m 
  [23/27648] dist=23084 m 
  [24/27648] dist=23069 m 
  [25/27648] dist=23154 m 
  [26/27648] dist=23139 m 
  [27/27648] dist=23155 m 
  [28/27648] dist=23140 m 
  [29/27648] dist=23154 m 
  [30/27648] dist=23139 m 
  [31/27648] dist=23155 m 
  [32/27648] dist=23140 m 
  [33/27648] dist=23154 m 
  [34/27648] dist=23139 m 
  [

## 4. Reconstrucción Topológica y Renderizado

El *solver* nos ha devuelto el orden lógico (ej. 0 $\rightarrow$ 3 $\rightarrow$ 1). Ahora recuperamos los nodos intermedios de las carreteras (usando `paths_dict`) para dibujar la ruta poligonal exacta sobre el mapa.

In [8]:
# Reconstruimos puntos_interes para la mejor combinación: nombre → [lon, lat] del parking elegido
puntos_interes = {}
for nombre, nodo_elegido in zip(nombres, nodos_objetivo):
    for pt in candidatos_utm[nombre]:
        if pt_to_nodo[pt] == nodo_elegido:
            lon, lat = pt_to_lonlat[pt]
            puntos_interes[nombre] = [lon, lat]
            break

# Reconstruimos la ruta completa a partir de los caminos pre-calculados
full_path_nodes = []
for i in range(len(optimal_order) - 1):
    u_idx = optimal_order[i]
    v_idx = optimal_order[i + 1]
    nodo_u = nodos_objetivo[u_idx]
    nodo_v = nodos_objetivo[v_idx]
    segment = paths_entre_nodos[(nodo_u, nodo_v)]
    if i > 0:
        segment = segment[1:]
    full_path_nodes.extend(segment)

# G_metros preserva 'lat' y 'lon' originales en cada nodo
coordenadas_ruta = [[G_metros.nodes[nodo]['lat'], G_metros.nodes[nodo]['lon']] for nodo in full_path_nodes]

# Dibujamos el mapa interactivo
mapa = folium.Map(location=[43.350, -3.852], zoom_start=14, tiles='cartodbpositron')

# Línea de la ruta
folium.PolyLine(coordenadas_ruta, color="#FF5A5F", weight=5, opacity=0.8).add_to(mapa)

# Mapeamos cada recinto a su número de visita (posición en la ruta, 1-based)
nombre_a_orden = {nombres[idx]: pos + 1 for pos, idx in enumerate(optimal_order[:-1])}

# Agrupamos los recintos por coordenadas de parking para evitar marcadores superpuestos
coords_a_nombres = defaultdict(list)
for nombre, coords in puntos_interes.items():
    coords_a_nombres[tuple(coords)].append(nombre)

for coords, nombres_grupo in coords_a_nombres.items():
    es_entrada = "Entrada Principal" in nombres_grupo
    bg_color = '#2ca02c' if es_entrada else '#1f77b4'

    nums = sorted(nombre_a_orden[n] for n in nombres_grupo)
    num_label = "/".join(str(n) for n in nums)

    popup_texto = "<br>".join(f"<b>{n}</b>" for n in nombres_grupo)

    folium.Marker(
        location=[coords[1], coords[0]],
        popup=folium.Popup(popup_texto, max_width=200),
        icon=folium.DivIcon(
            html=f'''<div style="
                background-color: {bg_color};
                color: white;
                border-radius: 50%;
                width: 28px;
                height: 28px;
                display: flex;
                align-items: center;
                justify-content: center;
                font-weight: bold;
                font-size: 12px;
                border: 2px solid white;
                box-shadow: 1px 1px 3px rgba(0,0,0,0.5);
            ">{num_label}</div>''',
            icon_size=(28, 28),
            icon_anchor=(14, 14)
        )
    ).add_to(mapa)

# Guardamos y abrimos el mapa en el navegador
output_path = os.path.abspath('../data/processed/ruta_optima.html')
mapa.save(output_path)
webbrowser.open(f'file://{output_path}')
print(f"Mapa guardado en: {output_path}")


Mapa guardado en: /Users/david/Documents/cabarceno-routing-optimization/data/processed/ruta_optima.html
